# Beat and signal-quality extraction

Detects cardiac cycles and arterial/PPG pulse structure, records quality masks, and verifies stored detections.

Run the setup cell first. Full processing and training require locally provisioned datasets; the grading quick start uses synthetic tests and saved aggregate results.


In [ ]:
from pathlib import Path
import sys, json, os, time, shutil
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'hemodynamics.json').exists())
assert Path(sys.executable).resolve() == (ROOT / '.venv' / 'Scripts' / 'python.exe').resolve(), 'Select the project .venv kernel'
sys.path.insert(0, str(ROOT / 'notebooks'))
import importlib, psutil
import numpy as np, pandas as pd
from IPython.display import Image, display
from hemodynamics import snapshot, beats, beats_pipeline as P, beats_qc as QC
for _module in (snapshot, beats, P, QC):   # pick up edited code in a long-lived kernel
    importlib.reload(_module)

me = psutil.Process()
for p in psutil.process_iter():
    try: p.cpu_percent(None)
    except psutil.Error: pass
time.sleep(0.8)
busy = []
for p in psutil.process_iter(['pid', 'name']):
    try:
        if p.pid != me.pid and p.info['name'] and 'python' in p.info['name'].lower() and p.cpu_percent(None) > 50:
            busy.append(p.pid)
    except psutil.Error:
        pass
locks = [str(p.relative_to(ROOT)) for p in (ROOT / 'data' / 'checkpoints').rglob('*.lock')]
settings = json.loads((ROOT / 'configs' / 'hemodynamics.json').read_text())
cfg = P.load_beats_config(settings)
print('Root:', ROOT)
print(f'Free disk: {shutil.disk_usage(ROOT).free / 1e9:.0f} GB | free RAM: {psutil.virtual_memory().available / 1e9:.0f} GB | CPUs: {os.cpu_count()}')
print('Other busy python processes:', busy or 'none', '| run locks:', locks or 'none')
print('Dataset tag:', snapshot.assert_dataset_v1(ROOT)['version'], '| beats config:', {k: cfg[k] for k in ('version', 'workers', 'phase_samples', 'store_waveforms', 'min_valid_frac')}, '| code', P.BEATS_CODE_VERSION)
if busy or locks:
    print('WARNING: another pipeline looks active. Wait for it to finish before section 4 (a stale lock from a crash is cleared automatically).')

## 2. Detection quality check on 12 cases (6 VitalDB + 6 MOVER)
The cases are chosen deterministically (one low-plausibility case per dataset plus five random ones). The check is run on the **stored** output, so what you see is exactly what the full run will write. Gates are evaluated as the median over the 12 cases per dataset:

| check | gate | why |
|---|---|---|
| ART pulse found for a valid R beat | >= 0.90 VitalDB, >= 0.80 MOVER | MOVER arterial channels are noisier (8 s on / 2 s off capture) |
| ECG heart rate vs ART heart rate | <= 3 bpm | two independent detectors must agree |
| ECG->ART delay | 0.05-0.40 s | physiological pulse arrival |
| ECG->PPG delay | 0.10-1.00 s | **widened from 0.50 s**: real oximeter latency is about 0.65 s (VitalDB) and 0.40 s (MOVER) |
| abs(ART->PPG delay) | <= 0.80 s | **widened from 0.35 s** for the same reason |
| IQR of the ECG->ART delay | <= 0.08 s | a stable delay means the same fiducial is found every beat |
| R markers on the dominant ECG deflection | >= 0.90 | independent of the detector, catches a constant offset onto the T wave or undershoot |

Section 4 refuses to start unless this passes (or you set `FORCE = True`).

In [ ]:
report, per_case, gate_table = QC.run_qc(ROOT, figures=True, progress=True)
show = ['dataset', 'case_id', 'beats', 'valid_frac', 'art_align_rate', 'ppg_align_rate', 'hr_diff_bpm', 'ptt_ecg_art', 'ptt_ecg_ppg', 'ptt_art_ppg',
        'ptt_ecg_art_iqr', 'sqi_ecg', 'sqi_art', 'sqi_ppg', 'r_peak_alignment']
display(per_case[show].round(3))
display(gate_table)
print('QC PASSED' if report['qc_passed'] else 'QC FAILED: do not run section 4; send results/beats_qc.json and the figures back', '|', len(report['figures']), 'overlay figures in figures/beats_qc/')

**Look at the overlays.** Each shows ~15 s of ECG, ART and PPG with the *stored* detections (red triangles = R-peaks and ART systolic peaks, green = pulse feet) and the per-beat SQI strip. Red markers must sit on the QRS spikes, the green ART foot at the base of the upstroke, the PPG foot at the base of the pulse upstroke. Gaps and flat segments stay unmarked on purpose.

In [ ]:
for ds in ('vitaldb', 'mover'):
    pngs = [ROOT / p for p in report['figures'] if Path(p).name.startswith(ds + '_')]
    for png in pngs[:2] + pngs[3:5]:
        print(png.name); display(Image(filename=str(png)))

## 3. Projection of size and time for all cases
Runs 40 cases (the 12 above plus 28 more) and extrapolates. If the projected waveform output exceeds `beats.max_projected_waveform_gb` (60 GB), phase samples are halved to 100, then waveforms are dropped, and the config is edited **additively** (only the `beats` keys). If that happens, re-run section 2 before section 4.

In [ ]:
dry = QC.pick_dry_run(ROOT, QC.pick_cases(ROOT))
dry_result = P.run(ROOT, case_ids=dry, progress=True)
proj = P.project(ROOT)
budget = P.apply_storage_budget(ROOT, proj)
free = shutil.disk_usage(ROOT).free / 1e9
display(pd.Series({
    'sample cases': proj['sample_cases'], 'beats per ready hour': proj['beats_per_ready_hour'], 'projected beats (all cases)': f"{proj['projected_total_beats']:,}",
    'projected scalar table GB': proj['projected_scalar_gb'], 'projected waveform GB': proj['projected_waveform_gb'], 'projected TOTAL GB': proj['projected_total_gb'],
    'still to write GB': proj['remaining_gb_estimate'], 'free disk GB now': round(free), 'free after run GB (est.)': round(free - proj['remaining_gb_estimate']),
    'estimated wall minutes (extrapolation)': proj['projected_wall_minutes_extrapolation'], 'workers': proj['workers']}, name='projection').to_frame())
print('Storage budget:', budget)
if budget['changed']:
    print('The config changed: re-run section 2 and section 3 once more before section 4.')
if free - proj['remaining_gb_estimate'] < cfg['min_free_after_gb']:
    print(f"WARNING: less than {cfg['min_free_after_gb']} GB would remain free. Free space first (the run also stops itself before filling the disk).")

## 4. Full beat extraction
- Uses `beats.workers` (20) processes, one case per task, biggest cases first. Finished cases are skipped on a rerun, so **resuming = run this same cell again**.
- To pause: press **Interrupt once**. The cases in flight finish and the status prints `PAUSED`. (Interrupting twice abandons them; they are redone on resume.)
- Status meanings: `COMPLETE` all cases done; `PAUSED` you interrupted or the disk guard stopped it (free space, rerun); `INCOMPLETE` some cases failed (listed in `data/manifests/beats_errors.csv`, rerun retries them); `FAILED` refused to start (see message).
- The dataset guard refuses to run if the unified index changed after the v1 tag.

In [ ]:
FORCE = False   # True only if you knowingly want to run although the QC gate below is not satisfied

qc_report = json.loads((ROOT / 'results' / 'beats_qc.json').read_text()) if (ROOT / 'results' / 'beats_qc.json').exists() else {}
chash = P.config_hash(P.load_beats_config(json.loads((ROOT / 'configs' / 'hemodynamics.json').read_text())))
qc_keys = {(d, str(c)) for d, c in qc_report.get('cases', [])}
man = P.collect_manifest(ROOT, write=False)
man_qc = man.loc[[(d, str(c)) in qc_keys for d, c in zip(man.dataset, man.case_id)]] if len(man) else man
qc_current = len(man_qc) == len(qc_keys) > 0 and set(man_qc.config_hash) == {chash}
if (qc_report.get('qc_passed') and qc_current) or FORCE:
    full = P.run(ROOT, progress=True)
    display(pd.Series(full, name='run').to_frame())
else:
    print('NOT STARTED: QC has not passed for the current configuration (qc_passed=%s, qc_matches_config=%s). Re-run section 2 (and 3), or set FORCE = True.' % (qc_report.get('qc_passed'), qc_current))

## 5. Summary tables
Per-case medians from the manifest, joined to the unified index. The MOVER vs VitalDB rows are the **domain shift** to keep in mind for the cross-dataset experiments (different oximeter latency, different arterial-signal quality).

In [ ]:
manifest = P.collect_manifest(ROOT)
index = pd.read_parquet(ROOT / 'data' / 'manifests' / 'unified_case_index.parquet')[['dataset', 'case_id', 'split_pooled', 'split_within_dataset']].astype({'case_id': str})
m = manifest.astype({'case_id': str}).merge(index, on=['dataset', 'case_id'], how='left')
m['GB'] = (m.bytes_scalar + m.bytes_waveforms) / 1e9
by = m.groupby(['dataset', 'split_pooled']).agg(cases=('case_id', 'size'), beats=('n_beats', 'sum'), valid_beat_frac=('valid_frac', 'mean'), art_ok=('art_ok_frac', 'mean'),
        ppg_ok=('ppg_ok_frac', 'mean'), sqi_ecg=('sqi_ecg_median', 'median'), sqi_art=('sqi_art_median', 'median'), sqi_ppg=('sqi_ppg_median', 'median'), GB=('GB', 'sum')).round(3)
display(by)
dist = m.groupby('dataset')[['hr_median', 'ptt_ecg_art_median', 'ptt_ecg_ppg_median', 'sqi_ecg_median', 'sqi_art_median', 'sqi_ppg_median']].describe(percentiles=[.1, .5, .9]).T
display(dist.loc[(slice(None), ['count', '10%', '50%', '90%']), :].round(3))
print(f"disk used by beats output: {m.GB.sum():.2f} GB over {len(m)} cases, {int(m.n_beats.sum()):,} beats")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, col, title in zip(axes, ('ptt_ecg_art_median', 'ptt_ecg_ppg_median', 'sqi_art_median'), ('ECG->ART delay (s)', 'ECG->PPG delay (s)', 'ART SQI')):
    for ds, g in m.groupby('dataset'):
        ax.hist(g[col].dropna(), bins=30, alpha=.55, density=True, label=ds)
    ax.set_title(title, fontsize=9); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. Verify the outputs
All checks should print `True` after a COMPLETE run. If anything is `False`, send back the printed lines and `data/manifests/beats_errors.csv`.

In [ ]:
wins = {}
for ds in ('mover', 'vitaldb'):
    w = pd.read_parquet(ROOT / 'data' / 'manifests' / f'ready_windows_{ds}.parquet', columns=['case_id', 'min_valid_frac', 'is_ready'])
    wins[ds] = set(w.loc[w.is_ready & (w.min_valid_frac >= cfg['min_valid_frac'] - 1e-6), 'case_id'].astype(str))
index_all = pd.read_parquet(ROOT / 'data' / 'manifests' / 'unified_case_index.parquet')
expected = {(r.dataset, str(r.case_id)) for r in index_all.itertuples() if str(r.case_id) in wins[r.dataset]}
got = {(d, str(c)) for d, c in zip(manifest.dataset, manifest.case_id)}
errors_path = ROOT / 'data' / 'manifests' / 'beats_errors.csv'
chash = P.config_hash(P.load_beats_config(json.loads((ROOT / 'configs' / 'hemodynamics.json').read_text())))
sample = manifest.sample(min(15, len(manifest)), random_state=1) if len(manifest) else manifest
loadable = all(len(P.load_beats(ROOT, r.dataset, r.case_id, columns=['t_r'])) == r.n_beats for r in sample.itertuples()) if len(sample) else False
checks = {
    'cases with ready windows (expected in manifest)': len(expected),
    'manifest rows': len(manifest),
    'every expected case is in the manifest': expected <= got,
    'all rows use the current beats config': set(manifest.config_hash) == {chash} if len(manifest) else False,
    'sample of per-case files loads and matches the manifest': loadable,
    'beat_waveforms present for sample': all((P.case_dir(ROOT, r.dataset, r.case_id) / 'beat_waveforms.parquet').exists() for r in sample.itertuples()) if (len(sample) and cfg['store_waveforms']) else True,
    'unfinished or failed cases still listed in beats_errors.csv': (len(pd.read_csv(errors_path).drop_duplicates(['dataset', 'case_id'])) if errors_path.exists() else 0),
    'dataset still v1': snapshot.assert_dataset_v1(ROOT)['version'],
}
for k, v in checks.items(): print(f'{k}: {v}')
print('FULL RUN VERIFIED' if checks['every expected case is in the manifest'] and checks['all rows use the current beats config'] and loadable else 'NOT COMPLETE YET: run section 4 again (it resumes)')